In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from src.db import get_engine

In [3]:
df = pd.read_parquet("/Users/emrekaya/PycharmProjects/eu_pesticide_mrl_compliance/data/processed/samples_raw.parquet")

In [4]:
df.head()

,sampId_A,sampCountry,origCountry,sampY,sampM,sampD,sampMatCode.base.building,paramCode.base.param,resVal,resLOQ,resType,evalCode
0,093B60FD0557804C8BA0CBF1453DA22F,ES,ES,2020,7,2,A031G,RF-0239-002-PPP,<NA>,0.005,LOQ,J002A
1,093B60FD0557804C8BA0CBF1453DA22F,ES,ES,2020,7,2,A031G,RF-0120-001-PPP,<NA>,0.005,LOQ,J002A
2,2056D8C1DEC3D12CBCE646B348D189D1,ES,ES,2020,12,17,A0DVE,RF-0079-001-PPP,<NA>,0.01,LOQ,J002A
3,2056D8C1DEC3D12CBCE646B348D189D1,ES,ES,2020,12,17,A0DVE,RF-0238-001-PPP,<NA>,0.01,LOQ,J002A
4,2056D8C1DEC3D12CBCE646B348D189D1,ES,ES,2020,12,17,A0DVE,RF-0408-001-PPP,<NA>,0.01,LOQ,J002A


In [5]:
df.shape

(32387997, 12)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32387997 entries, 0 to 32387996
Data columns (total 12 columns):
 #   Column                     Dtype          
---  ------                     -----          
 0   sampId_A                   string         
 1   sampCountry                string         
 2   origCountry                string         
 3   sampY                      int64[pyarrow] 
 4   sampM                      int64[pyarrow] 
 5   sampD                      int64[pyarrow] 
 6   sampMatCode.base.building  string         
 7   paramCode.base.param       string         
 8   resVal                     double[pyarrow]
 9   resLOQ                     double[pyarrow]
 10  resType                    string         
 11  evalCode                   string         
dtypes: double[pyarrow](2), int64[pyarrow](3), string(7)
memory usage: 4.8 GB


In [7]:
df.columns #I ll rename columns names for easy to understand

Index(['sampId_A', 'sampCountry', 'origCountry', 'sampY', 'sampM', 'sampD',
       'sampMatCode.base.building', 'paramCode.base.param', 'resVal', 'resLOQ',
       'resType', 'evalCode'],
      dtype='str')

In [8]:
df.isna().sum()
# there is missing values

sampId_A                            0
sampCountry                         0
origCountry                     12164
sampY                               0
sampM                               0
sampD                               0
sampMatCode.base.building           0
paramCode.base.param                0
resVal                       32247662
resLOQ                          26141
resType                             0
evalCode                            0
dtype: int64

In [9]:
df["resVal"].value_counts()

resVal
0.011     3885
0.01      3615
0.012     3613
0.013     3247
0.014     2940
          ... 
2.449        1
2.043        1
1.673        1
98.0         1
0.3499       1
Name: count, Length: 6453, dtype: int64[pyarrow]

In [10]:
df.groupby("resType")["resVal"]

In [11]:
df["resLOQ"].value_counts()

resLOQ
0.01      25638228
0.001      1674957
0.02       1608233
0.005      1184185
0.002       609974
            ...   
0.0388           1
0.0394           1
0.052            1
0.0414           1
0.0392           1
Name: count, Length: 2655, dtype: int64[pyarrow]

In [12]:
df[df["resLOQ"].isna()]["resType"].value_counts

<bound method IndexOpsMixin.value_counts of 573589      LOQ
576442      LOQ
588524      LOQ
592398      LOQ
644094      BIN
           ... 
29669628    LOD
29669700    LOD
29730737    LOD
29730839    LOD
29730890    LOD
Name: resType, Length: 26141, dtype: string>

In [13]:
df["origCountry"].value_counts()
# There is XX at origCountry

origCountry
DE    8826889
ES    4603772
FR    3699763
XX    3306952
IT    1609565
       ...   
MO        189
XK         54
HK          4
PS          4
SG          1
Name: count, Length: 146, dtype: int64[pyarrow]

In [14]:
# Cleaning

# adding new EU countries

# rename column names

# fixing data types (numeric and int columns)(The pyarrow engine already inferred correct types on read numeric columns,that's why I ll not fix .)

# build a single date column.

# Standardise missing origin country(there is missing values and XX)

#resVal: empty in 32.2M rows, but only filled when result_type ="VAL". Structurally empty, not missing data.
# resLoq: empty in 26k rows, 99% of them BIN results, which are binary
#   detected,not-detected outcomes with no quantification limit.
# Neither is imputed filling them would invent measurements that were never taken.

# Add derived flags (evaluated, exceeds_mrl, non_compliant, is_eu_origin) because  EFSA's evaluation codes into simple true/false columns so the logic is defined once here instead of repeated in every query and chart.

In [15]:
#####

In [16]:
# new countries

In [17]:
EU_COUNTRIES = [
    "AT","BE","BG","HR","CY","CZ","DK","EE","FI","FR","DE","GR","HU",
    "IE","IT","LV","LT","LU","MT","NL","PL","PT","RO","SK","SI","ES","SE",
]

In [18]:
df["is_eu"] = df["origCountry"].isin(EU_COUNTRIES)

In [19]:
# Rename column names

In [20]:
RENAME = {
    "sampId_A": "sample_id",
    "sampCountry": "reporting_country",
    "origCountry": "origin_country",
    "sampY": "sample_year",
    "sampM": "sample_month",
    "sampD": "sample_day",
    "sampMatCode.base.building": "product_code",
    "paramCode.base.param": "substance_code",
    "resVal": "result_value",
    "resLOQ": "loq",
    "resType": "result_type",
    "evalCode": "eval_code",
}

In [21]:
df = df.rename(columns=RENAME)

In [22]:
df.columns # column names renamed.

Index(['sample_id', 'reporting_country', 'origin_country', 'sample_year',
       'sample_month', 'sample_day', 'product_code', 'substance_code',
       'result_value', 'loq', 'result_type', 'eval_code', 'is_eu'],
      dtype='str')

In [23]:
###

In [24]:
# Build a single date column

In [25]:
dates = df[["sample_year", "sample_month", "sample_day"]]
dates.columns = ["year", "month", "day"]

In [26]:
df["sample_date"] = pd.to_datetime(dates, errors="coerce")


In [27]:
print("Invalid dates:", df["sample_date"].isna().sum())
# there is no invalid dates at data.

Invalid dates: 0


In [28]:
###

In [29]:
# Standardise missing origin country

In [30]:
df["origin_country"].isna().sum()

np.int64(12164)

In [31]:
df["origin_country"].value_counts()
# XX

origin_country
DE    8826889
ES    4603772
FR    3699763
XX    3306952
IT    1609565
       ...   
MO        189
XK         54
HK          4
PS          4
SG          1
Name: count, Length: 146, dtype: int64[pyarrow]

In [32]:
df["origin_country"] = df["origin_country"].fillna("UNKNOWN")

In [33]:
df["origin_country"] = df["origin_country"].replace("XX", "UNKNOWN")

In [34]:
print(df["origin_country"].isna().sum())
print(df["origin_country"].value_counts())
# missing values and XX fixed.

0
origin_country
DE         8826889
ES         4603772
FR         3699763
UNKNOWN    3319116
IT         1609565
            ...   
MO             189
XK              54
HK               4
PS               4
SG               1
Name: count, Length: 146, dtype: int64[pyarrow]


In [35]:
###

In [36]:
#Add derived flags

In [37]:
df["evaluated"] = df["eval_code"] != "J029A" #J029A = result not evaluated
df["exceeds_mrl"] = df["eval_code"].isin(["J003A", "J031A"]) #above the legal limit
df["non_compliant"] = df["eval_code"] == "J003A" #above the limit beyond measurement uncertainty

In [38]:
# Group origins into EU / non EU / unknown .The main dimension of the analysis
df["origin_group"] = "NON_EU"
df.loc[df["origin_country"].isin(EU_COUNTRIES), "origin_group"] = "EU"
df.loc[df["origin_country"] == "UNKNOWN", "origin_group"] = "UNKNOWN"

In [39]:
df["origin_group"].value_counts() #  I ll search unknown ,which country has highest one.

origin_group
EU         20902714
NON_EU      8166167
UNKNOWN     3319116
Name: count, dtype: int64

In [40]:
# At this project row counts look inflated, the real size is the number of samples . That's why I ll check samples always.

In [41]:
#checking sample level
df.groupby("origin_group")["sample_id"].nunique()


origin_group
EU         70886
NON_EU     22801
UNKNOWN    10134
Name: sample_id, dtype: int64

In [42]:
#
unknown = df[df["origin_group"] == "UNKNOWN"]

In [43]:
unknown.groupby("reporting_country")["sample_id"].nunique()
# Missing origin is concentrated in Germany: 9,441 of its samples, approximately 17% of its total.

reporting_country
DE    9441
ES      41
FR     652
Name: sample_id, dtype: int64

In [44]:
# Validation
print("Rows:", len(df))

Rows: 32387997


In [45]:
evaluated = df[df["evaluated"] == True]
samples = evaluated.groupby("sample_id")["exceeds_mrl"].max()

In [46]:
rate = samples.mean() * 100
print("Exceedance rate %:", round(rate, 2))

Exceedance rate %: 3.6


In [47]:
# saving clean dataset

In [48]:
PROCESSED = Path("/Users/emrekaya/PycharmProjects/eu_pesticide_mrl_compliance/data/processed")

In [49]:
df.to_parquet(PROCESSED / "samples_clean.parquet", index=False)
print("Saved:", len(df), "rows")

Saved: 32387997 rows


In [50]:
#checking for logic questions

In [51]:
# can one sample have more than one product?
counts = df.groupby("sample_id")["product_code"].nunique()
print(counts.value_counts())


product_code
1    103817
2         3
Name: count, dtype: int64


In [52]:
# Only 3 samples of 103,820 have two products. This also makes their substance rows look duplicated. Too small to matter left as is.

In [55]:
# Export for the Tableau dashboard

OUT = Path("../data/tableau") / "data" / "tableau"

QUERY = """
SELECT
    s.sample_id,
    s.reporting_country,
    s.origin_country,
    s.origin_group,
    s.sample_date,
    s.sample_year,
    s.sample_month,
    p.product_name,
    s.n_analyses,
    s.n_evaluated,
    s.exceeds_mrl,
    s.non_compliant
FROM fact_samples s
LEFT JOIN dim_product p ON s.product_code = p.product_code
WHERE s.n_evaluated > 0
"""


def main():
    OUT.mkdir(parents=True, exist_ok=True)

    engine = get_engine()
    df = pd.read_sql(QUERY, engine)

    df.to_csv(OUT / "samples_for_tableau.csv", index=False)
    print("rows:", len(df))


if __name__ == "__main__":
    main()

rows: 97584
